In [1]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader
from surprise import accuracy
from collections import defaultdict

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

reader = Reader(rating_scale=(1, 5))

print(f"Train: {len(train_df):,}개")
print(f"Test:  {len(test_df):,}개")
print("로드 완료")

Train: 80,000개
Test:  20,000개
로드 완료


In [2]:
def apply_temporal_weight(df, lam):
    """
    평점에 시간 감쇠 가중치를 적용
    w(t) = exp(-λ · (t_max - t) / t_half)
    오래된 평점일수록 가중치가 낮아짐
    """
    t_max  = df['timestamp'].max()
    t_min  = df['timestamp'].min()
    t_half = (t_max - t_min) / 2

    weights = np.exp(-lam * (t_max - df['timestamp']) / (t_half + 1))
    df = df.copy()
    df['weight'] = weights
    return df

# 확인 — λ=0.01 적용 시 가중치 분포
sample = apply_temporal_weight(train_df, lam=0.01)
print("가중치 통계:")
print(f"  최솟값 (가장 오래된 평점): {sample['weight'].min():.4f}")
print(f"  최댓값 (가장 최근 평점):   {sample['weight'].max():.4f}")
print(f"  평균:                      {sample['weight'].mean():.4f}")

가중치 통계:
  최솟값 (가장 오래된 평점): 0.9802
  최댓값 (가장 최근 평점):   1.0000
  평균:                      0.9895


In [3]:
class TemporalSVD:
    """
    시간 감쇠 가중치를 적용한 SVD
    가중치가 높은 평점(최근)이 학습에 더 큰 영향을 미침
    """
    def __init__(self, lam=0.01, n_factors=100, n_epochs=20,
                 lr_all=0.005, reg_all=0.02, random_state=42):
        self.lam = lam
        self.svd_params = dict(n_factors=n_factors, n_epochs=n_epochs,
                               lr_all=lr_all, reg_all=reg_all,
                               random_state=random_state)

    def fit(self, train_df, reader):
        # 시간 감쇠 가중치 적용
        weighted_df = apply_temporal_weight(train_df, self.lam)

        # 가중치 기반 샘플링 — 가중치 높은 평점을 더 많이 학습에 포함
        sampled_df = weighted_df.sample(
            n=len(weighted_df),
            weights='weight',
            replace=True,
            random_state=42
        )[['userId', 'movieId', 'rating']]

        dataset = Dataset.load_from_df(sampled_df, reader)
        trainset = dataset.build_full_trainset()

        self.model = SVD(**self.svd_params)
        self.model.fit(trainset)
        return self

    def test(self, test_df):
        testset = [
            (row['userId'], row['movieId'], row['rating'])
            for _, row in test_df.iterrows()
        ]
        return self.model.test(testset)

print("TemporalSVD 클래스 정의 완료")

TemporalSVD 클래스 정의 완료


In [4]:
lambda_list = [0.001, 0.005, 0.01, 0.05, 0.1]
results = []

print("λ 실험 시작 (5개 × 약 10초 = 총 1분 내외)\n")

for lam in lambda_list:
    model = TemporalSVD(lam=lam)
    model.fit(train_df, reader)
    predictions = model.test(test_df)

    rmse = accuracy.rmse(predictions, verbose=False)
    mae  = accuracy.mae(predictions,  verbose=False)

    results.append({'lambda': lam, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4)})
    print(f"λ={lam:<6} → RMSE: {rmse:.4f}  MAE: {mae:.4f}")

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['RMSE'].idxmin()]
print(f"\n최적 λ: {best['lambda']}  (RMSE: {best['RMSE']})")
print(f"베이스라인 SVD RMSE: 1.0300")
print(f"개선폭: {1.0300 - best['RMSE']:.4f}")

λ 실험 시작 (5개 × 약 10초 = 총 1분 내외)

λ=0.001  → RMSE: 1.0380  MAE: 0.8278
λ=0.005  → RMSE: 1.0351  MAE: 0.8244
λ=0.01   → RMSE: 1.0341  MAE: 0.8252
λ=0.05   → RMSE: 1.0356  MAE: 0.8269
λ=0.1    → RMSE: 1.0352  MAE: 0.8252

최적 λ: 0.01  (RMSE: 1.0341)
베이스라인 SVD RMSE: 1.0300
개선폭: -0.0041


In [5]:
class TemporalSVD_v2:
    """
    평점값 자체를 시간 가중치로 조정하는 방식
    최근 평점 → 전체 평균 쪽으로 덜 당겨짐 → 개인 취향 더 반영
    """
    def __init__(self, lam=0.01, n_factors=100, n_epochs=20,
                 lr_all=0.005, reg_all=0.02, random_state=42):
        self.lam = lam
        self.svd_params = dict(n_factors=n_factors, n_epochs=n_epochs,
                               lr_all=lr_all, reg_all=reg_all,
                               random_state=random_state)

    def fit(self, train_df, reader):
        weighted_df = apply_temporal_weight(train_df, self.lam).copy()

        # 전체 평균
        global_mean = weighted_df['rating'].mean()

        # 오래된 평점은 전체 평균 쪽으로 당김 (신뢰도 낮춤)
        weighted_df['rating_adjusted'] = (
            weighted_df['weight'] * weighted_df['rating'] +
            (1 - weighted_df['weight']) * global_mean
        )

        dataset = Dataset.load_from_df(
            weighted_df[['userId', 'movieId', 'rating_adjusted']]
            .rename(columns={'rating_adjusted': 'rating'}), reader
        )
        trainset = dataset.build_full_trainset()
        self.model = SVD(**self.svd_params)
        self.model.fit(trainset)
        return self

    def test(self, test_df):
        testset = [
            (row['userId'], row['movieId'], row['rating'])
            for _, row in test_df.iterrows()
        ]
        return self.model.test(testset)

# λ 그리드 탐색
lambda_list = [0.001, 0.005, 0.01, 0.05, 0.1]
results_v2 = []

print("v2 λ 실험 시작\n")
for lam in lambda_list:
    model = TemporalSVD_v2(lam=lam)
    model.fit(train_df, reader)
    predictions = model.test(test_df)

    rmse = accuracy.rmse(predictions, verbose=False)
    mae  = accuracy.mae(predictions,  verbose=False)

    results_v2.append({'lambda': lam, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4)})
    print(f"λ={lam:<6} → RMSE: {rmse:.4f}  MAE: {mae:.4f}")

results_v2_df = pd.DataFrame(results_v2)
best_v2 = results_v2_df.loc[results_v2_df['RMSE'].idxmin()]
print(f"\n최적 λ: {best_v2['lambda']}  (RMSE: {best_v2['RMSE']})")
print(f"베이스라인 SVD RMSE: 1.0300")
print(f"개선폭: {1.0300 - best_v2['RMSE']:.4f}")

v2 λ 실험 시작

λ=0.001  → RMSE: 1.0300  MAE: 0.8214
λ=0.005  → RMSE: 1.0300  MAE: 0.8214
λ=0.01   → RMSE: 1.0299  MAE: 0.8214
λ=0.05   → RMSE: 1.0295  MAE: 0.8216
λ=0.1    → RMSE: 1.0294  MAE: 0.8223

최적 λ: 0.1  (RMSE: 1.0294)
베이스라인 SVD RMSE: 1.0300
개선폭: 0.0006


In [6]:
import os
os.makedirs('../results', exist_ok=True)

# 최적 모델로 predictions 다시 생성
best_lam = best_v2['lambda']
final_model = TemporalSVD_v2(lam=best_lam)
final_model.fit(train_df, reader)
final_predictions = final_model.test(test_df)

rmse = accuracy.rmse(final_predictions, verbose=False)
mae  = accuracy.mae(final_predictions,  verbose=False)

# Precision, Recall, NDCG 측정
def precision_recall_ndcg_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions, recalls, ndcgs = {}, {}, {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]
        n_rel = sum(1 for (_, r) in user_ratings if r >= threshold)
        n_hit = sum(1 for (_, r) in top_k if r >= threshold)

        precisions[uid] = n_hit / k
        recalls[uid]    = n_hit / n_rel if n_rel > 0 else 0

        dcg  = sum((2**(r >= threshold) - 1) / np.log2(i + 2)
                   for i, (_, r) in enumerate(top_k))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(n_rel, k)))
        ndcgs[uid] = dcg / idcg if idcg > 0 else 0

    return (sum(precisions.values()) / len(precisions),
            sum(recalls.values())    / len(recalls),
            sum(ndcgs.values())      / len(ndcgs))

prec, rec, ndcg = precision_recall_ndcg_at_k(final_predictions)

# 비교표 출력
comparison = pd.DataFrame([
    {'model': 'SVD_baseline',   'lambda': '-',      'RMSE': 1.0300, 'MAE': 0.8214,
     'Precision@10': 0.6518, 'Recall@10': 0.4262, 'NDCG@10': 0.7823},
    {'model': 'TemporalSVD_v2', 'lambda': best_lam, 'RMSE': round(rmse,4), 'MAE': round(mae,4),
     'Precision@10': round(prec,4), 'Recall@10': round(rec,4), 'NDCG@10': round(ndcg,4)},
])

print(comparison.to_string(index=False))

comparison.to_csv('../results/temporal_svd_metrics.csv', index=False)
print("\n저장 완료 → results/temporal_svd_metrics.csv")

         model lambda   RMSE    MAE  Precision@10  Recall@10  NDCG@10
  SVD_baseline      - 1.0300 0.8214        0.6518     0.4262   0.7823
TemporalSVD_v2    0.1 1.0294 0.8223        0.6532     0.4271   0.7828

저장 완료 → results/temporal_svd_metrics.csv


In [7]:
movies_df = pd.read_csv('../data/raw/u.item', sep='|', encoding='latin-1',
                        names=['movieId','title','release_date','video_release',
                               'imdb_url','unknown','Action','Adventure','Animation',
                               'Children','Comedy','Crime','Documentary','Drama',
                               'Fantasy','Film-Noir','Horror','Musical','Mystery',
                               'Romance','Sci-Fi','Thriller','War','Western'],
                        usecols=['movieId','Action','Adventure','Animation',
                                 'Children','Comedy','Crime','Documentary','Drama',
                                 'Fantasy','Film-Noir','Horror','Musical','Mystery',
                                 'Romance','Sci-Fi','Thriller','War','Western'])

genre_cols = ['Action','Adventure','Animation','Children','Comedy','Crime',
              'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical',
              'Mystery','Romance','Sci-Fi','Thriller','War','Western']

movies_df = movies_df.set_index('movieId')

def intra_list_diversity(recommended_ids, genre_matrix, genre_cols):
    valid_ids = [i for i in recommended_ids if i in genre_matrix.index]
    if len(valid_ids) < 2:
        return 0
    vectors = genre_matrix.loc[valid_ids, genre_cols].values.astype(float)
    n = len(vectors)
    total = 0
    count = 0
    for i in range(n):
        for j in range(i+1, n):
            norm_i = np.linalg.norm(vectors[i])
            norm_j = np.linalg.norm(vectors[j])
            if norm_i > 0 and norm_j > 0:
                sim = np.dot(vectors[i], vectors[j]) / (norm_i * norm_j)
                total += (1 - sim)
                count += 1
    return total / count if count > 0 else 0

# 유저별 추천 top10 추출 후 ILD 계산
def compute_avg_ild(predictions, genre_matrix, genre_cols, k=10):
    user_est = defaultdict(list)
    for uid, iid, _, est, _ in predictions:
        user_est[uid].append((est, iid))

    ild_scores = []
    for uid, items in user_est.items():
        items.sort(key=lambda x: x[0], reverse=True)
        top_k_ids = [int(iid) for _, iid in items[:k]]
        ild = intra_list_diversity(top_k_ids, genre_matrix, genre_cols)
        ild_scores.append(ild)

    return np.mean(ild_scores)

# 베이스라인 ILD (baseline predictions 다시 생성)
from surprise import Dataset
baseline_data = Dataset.load_from_df(train_df[['userId','movieId','rating']], reader)
baseline_trainset = baseline_data.build_full_trainset()
baseline_svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
baseline_svd.fit(baseline_trainset)
baseline_testset = [(r['userId'], r['movieId'], r['rating']) for _, r in test_df.iterrows()]
baseline_preds = baseline_svd.test(baseline_testset)

ild_baseline = compute_avg_ild(baseline_preds, movies_df, genre_cols)
ild_temporal = compute_avg_ild(final_predictions, movies_df, genre_cols)

print(f"ILD — SVD 베이스라인:  {ild_baseline:.4f}")
print(f"ILD — TemporalSVD_v2: {ild_temporal:.4f}")
print(f"다양성 변화: {ild_temporal - ild_baseline:+.4f}")

ILD — SVD 베이스라인:  0.6500
ILD — TemporalSVD_v2: 0.6491
다양성 변화: -0.0008


In [8]:
final_comparison = pd.DataFrame([
    {'model': 'SVD_baseline',   'lambda': '-',
     'RMSE': 1.0300, 'MAE': 0.8214,
     'Precision@10': 0.6518, 'Recall@10': 0.4262,
     'NDCG@10': 0.7823, 'ILD': round(ild_baseline, 4)},
    {'model': 'TemporalSVD_v2', 'lambda': 0.1,
     'RMSE': 1.0294, 'MAE': 0.8223,
     'Precision@10': 0.6532, 'Recall@10': 0.4271,
     'NDCG@10': 0.7828, 'ILD': round(ild_temporal, 4)},
])

final_comparison.to_csv('../results/week2_final_metrics.csv', index=False)
print("=== 2주차 최종 결과 ===\n")
print(final_comparison.to_string(index=False))

=== 2주차 최종 결과 ===

         model lambda   RMSE    MAE  Precision@10  Recall@10  NDCG@10    ILD
  SVD_baseline      - 1.0300 0.8214        0.6518     0.4262   0.7823 0.6500
TemporalSVD_v2    0.1 1.0294 0.8223        0.6532     0.4271   0.7828 0.6491
